# Week 5 · Day 2  LangChain
### Tools, Chains, Memory & Your First Framework Agent (Gemini API)

Yesterday you built a ReAct agent by hand: a raw `while` loop, manual tool dispatch,
manually-threaded conversation history. Today you rebuild the same idea on
**LangChain**, then go further chaining prompts, adding real memory, adding a tool
that reads real external data, and forcing a structured final answer.

Every cell below calls the **real Gemini API** via `langchain google genai`.  You need a working API key before running this notebook.


## Step 0 — Get a Gemini API key


1. Go to [Google AI Studio](https://aistudio.google.com/apikey) and click
   **Create API key**. Copy it.
2. In this Colab notebook, click the **🔑 key icon** in the left sidebar.
3. Click **Add new secret**.
   - Name: `GEMINI_API_KEY`
   - Value: paste your key
4. Toggle **Notebook access** ON for this secret.


## Step 1 — Install LangChain + the Gemini integration

- `langchain-core` — the Runnable/LCEL primitives.
- `langchain-google-genai` — the Gemini chat-model wrapper (`ChatGoogleGenerativeAI`).
- `langchain-classic` — as of LangChain 1.x, the legacy agent constructors this task
  asks for (`create_tool_calling_agent`, `AgentExecutor`, `ConversationBufferMemory`)
  were split out of the core `langchain` package into `langchain-classic` for
  backwards compatibility. The modern replacement is `langgraph`'s `create_agent`,
  but we're deliberately using the classic API here since that's what the task
  specifies, and it maps most directly onto yesterday's raw loop.

Always install with `-U` (upgrade) -- Gemini's tool-calling protocol has changed
recently (see Step 9's note on `thought_signature`), and older package versions will
error on multi-step tool calls.


In [1]:
!pip install -q -U langchain-core langchain-google-genai langchain-classic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.5/734.5 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 137.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.2/320.2 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.0/134.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 152.3 MB/s eta 0:00:00


## Step 2 — Imports


In [2]:
import os
import json
import ast
import operator as op
from typing import List

from pydantic import BaseModel, Field

from google.colab import userdata

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.tools.base import ToolException
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_classic.agents import create_tool_calling_agent, AgentExecutor


## Step 3 — Load the API key and create the LLM wrapper


In [3]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it via the 🔑 key icon in the left sidebar "
        "(Add new secret -> name it GEMINI_API_KEY -> paste your key -> enable "
        "Notebook access), then re-run this cell."
    )

MODEL_NAME = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(model=MODEL_NAME, google_api_key=GEMINI_API_KEY)

print("LangChain Gemini chat model created successfully.")
print("Model:", MODEL_NAME)


LangChain Gemini chat model created successfully.
Model: gemini-3.5-flash-lite


## Step 4 — Sanity check: one plain call


In [4]:
response = llm.invoke("Say hello in exactly five words.")
print(response.content)


[{'type': 'text', 'text': 'Hello, how are you today?', 'extras': {'signature': 'EjQKMgERTTIPAONZmAlAbCLYwb2g/+qu+Z0RpWSar2Wh2sSNr6L+H7jT2tsE95iH7A8poWp8'}}]


## Task 1 : LangChain Setup & Core Concepts

### Mapping Day 1's raw concepts onto LangChain

| Day 1 (raw Python) | LangChain equivalent | What changed |
|---|---|---|
| `client.models.generate_content(...)` | `ChatGoogleGenerativeAI` (an **LLM wrapper**) | Same underlying Gemini API call, but wrapped in a standard `Runnable` interface shared across every model provider LangChain supports. `.invoke()`, `.stream()`, `.batch()` all work the same way regardless of which LLM is behind it. |
| A `TOOL_SCHEMAS` dict + a `TOOL_EXECUTORS` dict, kept in sync by hand | A single `@tool` decorated Python function | LangChain derives the JSON schema *automatically* from the function's type hints and docstring, instead of you writing the schema and the implementation as two separate things that can drift out of sync. |
| `run_agent()`  our handwritten `while` loop | `create_tool_calling_agent(...)` + `AgentExecutor` | The Reason → Act → Observe loop, the `max_iterations` guardrail, and the tool dispatch are all implemented for you inside `AgentExecutor`. |
| `contents` list we manually appended to every turn | `Memory` (`ConversationBufferMemory` / `RunnableWithMessageHistory`) | Conversation history is still just a list of messages under the hood LangChain just gives you a class that manages appending to it and re-injecting it into the prompt automatically. |

### LCEL and the pipe (`|`) operator

Every LangChain component a prompt template, a chat model, an output parser is a
`Runnable`: an object with a standard `.invoke()` / `.stream()` / `.batch()` interface.
The `|` operator is just `Runnable.__or__`, and it does one thing: build a
`RunnableSequence` that feeds the output of the left-hand Runnable directly into the
input of the right-hand Runnable when `.invoke()` is called. `prompt | llm | parser`
is exactly equivalent to `parser.invoke(llm.invoke(prompt.invoke(input)))`, just
expressed as a composable pipeline instead of nested calls and because every piece
implements the same interface, the *whole chain* automatically gets streaming,
batching, and async support for free, without any of the pieces needing to know about
each other.

### A simple LCEL chain: prompt → response


In [5]:
simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    ("human", "{question}"),
])

simple_chain = simple_prompt | llm | StrOutputParser()

result = simple_chain.invoke({"question": "In one sentence, what is a ReAct agent?"})
print(result)


A ReAct agent is an AI system that combines reasoning (planning and thinking) with acting (using tools and interacting with environments) in an interleaved loop to solve complex tasks.


## Task 2 : Define & Register Tools

Three tools: two reused from Day 1 (`calculator`, `get_weather`), and one new tool
that reads from a **real local data source** a small JSON "database" of laptop
products (`products.json`), used later for the Task 4 memory scenario ("find the
price of X, compare to Y, recommend to a budget conscious client").


### Step 5 — Create the product catalog (the real data source)


In [6]:
PRODUCTS_FILE = "products.json"

catalog = [
    {"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0,  "ram_gb": 8,  "storage_gb": 256,  "rating": 4.3},
    {"name": "UltraBook Pro 14", "category": "laptop", "price_usd": 1499.0, "ram_gb": 16, "storage_gb": 512,  "rating": 4.6},
    {"name": "UltraBook Pro 16", "category": "laptop", "price_usd": 1899.0, "ram_gb": 32, "storage_gb": 1024, "rating": 4.7},
    {"name": "ValueBook 14",     "category": "laptop", "price_usd": 599.0,  "ram_gb": 8,  "storage_gb": 256,  "rating": 4.0},
]

with open(PRODUCTS_FILE, "w") as f:
    json.dump(catalog, f, indent=2)

print(f"Wrote {len(catalog)} products to {PRODUCTS_FILE}")


Wrote 4 products to products.json


### Step 6 : Define the tools with `@tool`

**Why tool docstrings matter.** With `@tool`, the function's **docstring becomes the
tool's `description`** field, and the **type hints become the JSON schema**
(`input_schema`) both are sent to the model on every call, exactly like Day 1's
manually written `TOOL_SCHEMAS`. LangChain isn't doing anything magical here, it's
just deriving the same specification from your code instead of making you write it
twice. A vague docstring is just as damaging here as a vague `description` string was
on Day 1 it's still the entire spec the model uses to decide whether and how to
call the tool.


In [7]:
_SAFE_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Mod: op.mod, ast.Pow: op.pow, ast.USub: op.neg,
}


def _safe_eval(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Disallowed expression element: {ast.dump(node)}")


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the numeric result.
    Supports +, -, *, /, %, ** and parentheses. Use this any time the user
    asks for a numeric computation instead of computing it yourself, so the
    arithmetic is guaranteed correct. Example input: '(18 + 4) * 2'."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        return json.dumps({"success": True, "expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"success": False, "error": f"Could not evaluate '{expression}': {e}"})


print(calculator.invoke({"expression": "47 * 6 + 12"}))


{"success": true, "expression": "47 * 6 + 12", "result": 294}


In [8]:
_FAKE_WEATHER_DB = {
    "tokyo":  {"temp_c": 31, "condition": "humid, partly cloudy"},
    "paris":  {"temp_c": 22, "condition": "clear"},
    "lahore": {"temp_c": 34, "condition": "sunny"},
}


@tool
def get_weather(city: str) -> str:
    """Look up the current weather for a named city. Returns the
    temperature in Celsius and a short condition string. This is a
    stub/demo data source, not a live feed -- use it whenever the user
    asks about weather in a specific city."""
    key = city.strip().lower()
    if key not in _FAKE_WEATHER_DB:
        return json.dumps({"success": False, "error": f"No weather data available for '{city}' (demo dataset only)."})
    return json.dumps({"success": True, "city": city, **_FAKE_WEATHER_DB[key]})


print(get_weather.invoke({"city": "Lahore"}))


{"success": true, "city": "Lahore", "temp_c": 34, "condition": "sunny"}


In [9]:
@tool
def lookup_product_price(product_name: str) -> str:
    """Look up a product's price and specs from the local product catalog
    (products.json) by name. Matching is case-insensitive substring match,
    e.g. 'air 13' matches 'UltraBook Air 13'. Use this whenever the user
    asks for the price or specs of a named product. Returns an error if no
    product matches."""
    try:
        with open(PRODUCTS_FILE, "r") as f:
            products = json.load(f)
    except FileNotFoundError:
        return json.dumps({"success": False, "error": f"Catalog file '{PRODUCTS_FILE}' not found."})
    except Exception as e:
        return json.dumps({"success": False, "error": f"Could not read catalog: {e}"})

    key = product_name.strip().lower()
    matches = [p for p in products if key in p["name"].lower()]
    if not matches:
        return json.dumps({"success": False, "error": f"No product matching '{product_name}' found in catalog."})
    return json.dumps({"success": True, "matches": matches})


print(lookup_product_price.invoke({"product_name": "Air 13"}))


{"success": true, "matches": [{"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0, "ram_gb": 8, "storage_gb": 256, "rating": 4.3}]}


In [10]:
TOOLS = [calculator, get_weather, lookup_product_price]

for t in TOOLS:
    print(f"- {t.name}")
    print(f"    description: {t.description}")
    print(f"    args_schema: {t.args_schema.model_json_schema()['properties']}")
    print()


- calculator
    description: Evaluate a basic arithmetic expression and return the numeric result.
    Supports +, -, *, /, %, ** and parentheses. Use this any time the user
    asks for a numeric computation instead of computing it yourself, so the
    arithmetic is guaranteed correct. Example input: '(18 + 4) * 2'.
    args_schema: {'expression': {'title': 'Expression', 'type': 'string'}}

- get_weather
    description: Look up the current weather for a named city. Returns the
    temperature in Celsius and a short condition string. This is a
    stub/demo data source, not a live feed -- use it whenever the user
    asks about weather in a specific city.
    args_schema: {'city': {'title': 'City', 'type': 'string'}}

- lookup_product_price
    description: Look up a product's price and specs from the local product catalog
    (products.json) by name. Matching is case-insensitive substring match,
    e.g. 'air 13' matches 'UltraBook Air 13'. Use this whenever the user
    asks for th

## Task 3 : Build an Agent with `create_tool_calling_agent` / `AgentExecutor`

### Step 7 : Assemble the prompt

`create_tool_calling_agent` requires a `ChatPromptTemplate` with an `agent_scratchpad`
`MessagesPlaceholder` this is where `AgentExecutor` will insert the running record
of tool calls and their results (LangChain's equivalent of the `contents` list we
threaded by hand on Day 1). We also add a `chat_history` placeholder now, so the same
prompt is ready for Task 4's memory wiring.


In [11]:
SYSTEM_PROMPT = (
    "You are a careful shopping assistant with access to tools. Only call a "
    "tool when it is genuinely needed to answer the user, never invent a "
    "tool that isn't listed, and never fabricate a tool's result -- always "
    "wait for the tool result to be returned to you. If a tool returns an "
    "error, decide whether to retry, try a different approach, or tell the "
    "user you cannot complete the request. When you have enough "
    "information, answer in plain text with no further tool calls."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

print("Prompt assembled. Input variables:", prompt.input_variables)


Prompt assembled. Input variables: ['agent_scratchpad', 'input']


### Step 8 : Build the agent and the executor


In [12]:
agent = create_tool_calling_agent(llm, TOOLS, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,          # prints the full reasoning trace as it runs
    handle_parsing_errors=True,
    max_iterations=6,      # same guardrail idea as Day 1's max_iterations
)

print("AgentExecutor ready.")


AgentExecutor ready.


### Step 9 : Run a multi-step task and capture the trace

Note on Gemini 3.5 + tool calling: Gemini 3.5 attaches a `thought_signature` to every
`function_call` it returns, and requires that exact signature to be echoed back on
the next turn. `create_tool_calling_agent`'s scratchpad formatter (`format_to_tool_messages`)
handles this automatically it re uses the *exact* `AIMessage` object the model
returned (via `ToolAgentAction.message_log`) rather than rebuilding it from scratch,
which is precisely the pattern Google's docs require. This is one thing LangChain
gets right for free that Day 1's raw loop had to be patched by hand to fix.


In [13]:
result = agent_executor.invoke({
    "input": "Compare the price of the UltraBook Air 13 and the UltraBook Pro 14. "
             "Which one should I recommend to a budget-conscious client?"
})

print("\nFINAL OUTPUT:", result["output"])




> Entering new AgentExecutor chain...

Invoking: `lookup_product_price` with `{'product_name': 'UltraBook Air 13'}`


{"success": true, "matches": [{"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0, "ram_gb": 8, "storage_gb": 256, "rating": 4.3}]}
Invoking: `lookup_product_price` with `{'product_name': 'UltraBook Pro 14'}`


{"success": true, "matches": [{"name": "UltraBook Pro 14", "category": "laptop", "price_usd": 1499.0, "ram_gb": 16, "storage_gb": 512, "rating": 4.6}]}[{'type': 'text', 'text': 'Here is the price comparison between the two laptops:\n\n* **UltraBook Air 13:** $899 (8GB RAM, 256GB storage)\n* **UltraBook Pro 14:** $1,499 (16GB RAM, 512GB storage)\n\nFor a **budget-conscious client**, you should recommend the **UltraBook Air 13**. It is significantly more affordable at $899 (saving $600 compared to the Pro model) and offers a capable specification for everyday tasks and standard use.', 'index': 0, 'extras': {'signature': 'EjQKMgERTTIP76tU+jgubo5p

### Annotating the trace

Looking at the `verbose=True` output above, in order:

1. **Reason** (hidden) the model decides it needs the Air 13's price first. Nothing
   is printed for this step by default, only the *result* of the reasoning (the tool
   call) is shown.
2. **Act** `Invoking: 'lookup_product_price' with '{'product_name': 'UltraBook Air 13'}'`
3. **Observe** the tool's JSON output is printed right below the invocation.
4. **Reason** (hidden again) decides it still needs the Pro 14's price.
5. **Act** `Invoking: 'lookup_product_price' with '{'product_name': 'UltraBook Pro 14'}'`
6. **Observe** the second tool's JSON output.
7. **Reason → final answer** the model compares both prices and returns plain text,
   which `AgentExecutor` recognizes as the stopping condition (mirrors Day 1's "no
   function_call parts" check) and returns as `result["output"]`.

### Comparing this trace to Day 1's raw log

**Similar:** the actual shape of the loop is identical reason, act, observe,
repeat, stop on plain text. The tool calls and their JSON results are equally visible.

**Hidden now:** Day 1's `[reason]` lines printed the model's own words *between* tool
calls; `AgentExecutor`'s default verbose output does not show intermediate reasoning
text at all unless the model happens to include it as `AIMessage.content` alongside a
tool call. Also hidden: the exact prompt LangChain assembles from `ChatPromptTemplate`
(system + history + human + scratchpad, converted into the provider's specific
message format), and the internal `thought_signature` bookkeeping discussed above
Day 1 made you handle that by hand; here it's invisible unless something breaks.


## Task 4 : Add Memory

We use `RunnableWithMessageHistory` -- the "modern" LCEL-native memory approach the
task mentions as an alternative to `ConversationBufferMemory` -- wrapped around
`agent_executor`. It manages a per-session `InMemoryChatMessageHistory` and
automatically injects it into the `chat_history` placeholder in our prompt on every
call: LangChain's version of the `contents` list Day 1 threaded through
`run_agent()` by hand.

**One real wrinkle, fixed properly instead of worked around.** Wiring
`RunnableWithMessageHistory` directly around `agent_executor` (no normalization
step) works for Turn 1, then crashes on Turn 2 with:

```
ValueError: Message dict must contain 'role' and 'content' keys, got
{'type': 'text', 'text': 'The price of the UltraBook Air 13 is $899.',
 'index': 0, 'extras': {'signature': 'EjQKMgERTTIP...'}}
```

Gemini 3.5 sometimes returns the agent's final answer as a *list of content
blocks* (each carrying a `thought_signature` in `extras`) instead of a plain
string. `RunnableWithMessageHistory` only knows how to save a plain string (or an
already-built `list[BaseMessage]`) as the AI turn -- handed a list of raw
content-block dicts, it mistakes them for messages to save and tries to convert
each one, which fails immediately.

The fix below stays on the modern API instead of abandoning it: compose a small
`RunnableLambda` *between* `agent_executor` and `RunnableWithMessageHistory` that
normalizes the output to a plain string first. `RunnableWithMessageHistory` then
never has to deal with Gemini's content-block shape at all -- it saves history
exactly the way it would for any other model. See `normalize_agent_output`'s
docstring in Step 10 for the full explanation.

### Step 10 : Wire up session-based memory


In [14]:
# --- Step 10 ---
from langchain_core.runnables import RunnableLambda

_session_store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """RunnableWithMessageHistory calls this on every invoke to fetch (or
    lazily create) the message-history object for a given session_id."""
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()
    return _session_store[session_id]


def normalize_agent_output(result: dict) -> str:
    """AgentExecutor's `output` is normally a plain string, but Gemini 3.5
    sometimes returns it as a list of content blocks instead -- each block a
    dict with a 'text' field and a 'thought_signature' tucked into 'extras',
    e.g. [{'type': 'text', 'text': '...', 'extras': {'signature': '...'}}].

    This is exactly what breaks RunnableWithMessageHistory if it's wrapped
    directly around agent_executor with no normalization: Turn 1 works (no
    prior history to re-inject yet), then Turn 2 crashes with

        ValueError: Message dict must contain 'role' and 'content' keys, got
        {'type': 'text', 'text': 'The price of the UltraBook Air 13 is $899.',
         'index': 0, 'extras': {'signature': 'EjQKMgERTTIP...'}}

    RunnableWithMessageHistory only knows how to save a plain string (or an
    already-built list[BaseMessage]) as the AI turn. Handed a list of raw
    content-block dicts instead, it mistakes them for a list of *messages* to
    save and tries to convert each one, which has none of the fields a
    message needs.

    The fix is to normalize *before* RunnableWithMessageHistory ever sees the
    output, by composing this function into the chain with a RunnableLambda --
    so RunnableWithMessageHistory always receives a clean string and saves it
    the same way it would for any other model, regardless of provider."""
    output = result["output"]
    if isinstance(output, str):
        return output
    if isinstance(output, list):
        return "".join(block.get("text", "") for block in output if isinstance(block, dict))
    return str(output)


# agent_executor -> normalize its output to a plain string -> RunnableWithMessageHistory.
# Composing the normalization step *into* the chain (rather than post-processing
# after the fact, as the manual version of this notebook did) means
# RunnableWithMessageHistory does the actual history read/write -- it always
# receives a string, so it works exactly as documented.
agent_with_normalized_output = agent_executor | RunnableLambda(normalize_agent_output)

agent_with_history = RunnableWithMessageHistory(
    agent_with_normalized_output,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


def run_with_memory(session_id: str, user_input: str) -> str:
    """Convenience wrapper: RunnableWithMessageHistory takes session_id via
    `config`, not the input dict -- this just tucks that plumbing away so the
    rest of the notebook can call run_with_memory(session_id, text)."""
    return agent_with_history.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}},
    )


print("RunnableWithMessageHistory wired up: agent_executor -> normalize output -> history.")


RunnableWithMessageHistory wired up: agent_executor -> normalize output -> history.


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### Step 11 : Test a 3-turn conversation with follow-ups

Each turn uses the same `session_id`, so the agent must rely on `chat_history` alone
to resolve "it", "the other one", and "which one" in later turns there is no other
mechanism carrying that context forward.


In [15]:
# --- Step 11 (replacement) — same 3-turn scenario, same session id ---

session_id = "client-42"

turn_1_output = run_with_memory(session_id, "Find the price of the UltraBook Air 13.")
print("Turn 1:", turn_1_output)



> Entering new AgentExecutor chain...

Invoking: `lookup_product_price` with `{'product_name': 'UltraBook Air 13'}`


{"success": true, "matches": [{"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0, "ram_gb": 8, "storage_gb": 256, "rating": 4.3}]}[{'type': 'text', 'text': 'The UltraBook Air 13 is priced at $899.', 'index': 0, 'extras': {'signature': 'EjQKMgERTTIPAVcMzaOhkp93Guy+AinYfxJMGlGM/ywLEts7/rlTD/VGMWvydpzTdpEoNNpR'}}]

> Finished chain.
Turn 1: The UltraBook Air 13 is priced at $899.


In [16]:
turn_2_output = run_with_memory(session_id, "Now compare it to the UltraBook Pro 14.")
print("Turn 2:", turn_2_output)



> Entering new AgentExecutor chain...

Invoking: `lookup_product_price` with `{'product_name': 'UltraBook Air 13'}`


{"success": true, "matches": [{"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0, "ram_gb": 8, "storage_gb": 256, "rating": 4.3}]}
Invoking: `lookup_product_price` with `{'product_name': 'UltraBook Pro 14'}`


{"success": true, "matches": [{"name": "UltraBook Pro 14", "category": "laptop", "price_usd": 1499.0, "ram_gb": 16, "storage_gb": 512, "rating": 4.6}]}[{'type': 'text', 'text': 'Here is a comparison between the UltraBook Air 13 and the UltraBook Pro 14:\n\n* **UltraBook Air 13:**\n  * **Price:** $899\n  * **RAM:** 8 GB\n  * **Storage:** 256 GB\n  * **Rating:** 4.3 / 5\n\n* **UltraBook Pro 14:**\n  * **Price:** $1,499 (an increase of $600)\n  * **RAM:** 16 GB (double the Air)\n  * **Storage:** 512 GB (double the Air)\n  * **Rating:** 4.6 / 5\n\nThe Pro 14 offers double the RAM and storage, a slightly larger screen format (implied by the name), 

In [17]:
turn_3_output = run_with_memory(session_id, "Which one should I recommend to a budget-conscious client?")
print("Turn 3:", turn_3_output)



> Entering new AgentExecutor chain...
[{'type': 'text', 'text': 'For a budget-conscious client, you should recommend the **UltraBook Air 13**. \n\nAt $899, it is $600 cheaper than the UltraBook Pro 14 ($1,499). While the Pro 14 offers double the RAM (16GB vs 8GB) and storage (512GB vs 256GB), the Air 13 still provides plenty of capability for everyday tasks, office work, and web browsing while keeping costs significantly lower.', 'index': 0, 'extras': {'signature': 'EjQKMgERTTIPxzaaukAX4OcDUf151Zwt7hl5WzshG0CtK2RNK2Bmq8ZA2eSxKzzR8XhcxXbG'}}]

> Finished chain.
Turn 3: For a budget-conscious client, you should recommend the **UltraBook Air 13**. 

At $899, it is $600 cheaper than the UltraBook Pro 14 ($1,499). While the Pro 14 offers double the RAM (16GB vs 8GB) and storage (512GB vs 256GB), the Air 13 still provides plenty of capability for everyday tasks, office work, and web browsing while keeping costs significantly lower.


In [18]:
print("Full stored history for this session:")
for m in _session_store[session_id].messages:
    print(f"  [{m.type}] {m.content[:120]}")


Full stored history for this session:
  [human] Find the price of the UltraBook Air 13.
  [ai] The UltraBook Air 13 is priced at $899.
  [human] Now compare it to the UltraBook Pro 14.
  [ai] Here is a comparison between the UltraBook Air 13 and the UltraBook Pro 14:

* **UltraBook Air 13:**
  * **Price:** $899
  [human] Which one should I recommend to a budget-conscious client?
  [ai] For a budget-conscious client, you should recommend the **UltraBook Air 13**. 

At $899, it is $600 cheaper than the Ult


## Task 5 : Structured Output & Error Handling

### Step 12 : Force the final answer into a Pydantic schema

We define a `Recommendation` model and use `with_structured_output()` as a small
follow up chain that reads the agent's free text answer and extracts it into a typed
object. (We compose it *after* the agent rather than inside `AgentExecutor` itself,
because the agent's own tool calling loop and a single structured final answer are two
different jobs forcing them into one call would mean the model could no longer
freely call tools *and* return structured output in the same turn. Post processing the
final answer keeps both jobs clean.)


In [19]:
class Recommendation(BaseModel):
    """A structured shopping recommendation."""
    recommended_product: str = Field(description="Name of the recommended product")
    price_usd: float = Field(description="Price of the recommended product in USD")
    reasoning: str = Field(description="Why this product was recommended")
    alternative_considered: str = Field(description="The other product that was compared against")


structured_llm = llm.with_structured_output(Recommendation)

structured_result = structured_llm.invoke(
    "Based on this recommendation, extract the structured fields.\n\n"
    f"Recommendation text: {turn_3_output}\n\n"
    f"(For context, earlier in the conversation: {turn_2_output})"
)

print(type(structured_result))
print(structured_result)

<class '__main__.Recommendation'>
recommended_product='UltraBook Air 13' price_usd=899.0 reasoning='It is $600 cheaper than the Pro 14 and still provides plenty of capability for everyday tasks, office work, and web browsing for a budget-conscious client.' alternative_considered='UltraBook Pro 14'


### Step 13 : Error handling: a tool that sometimes throws

By default, a raw Python exception inside a `@tool` function **propagates all the way
up and crashes `AgentExecutor.invoke()`** LangChain only catches its own
`ToolException` type automatically, not arbitrary exceptions. So the tool itself has
to opt in: catch the real error, re raise it as a `ToolException`, and set
`handle_tool_error=True` on the tool object. That's the one piece of configuration
that turns a crash into a normal observation the agent can react to and retry.


In [20]:
_flaky_state = {"should_fail": True}


@tool
def flaky_lookup_product_price(product_name: str) -> str:
    """Same as lookup_product_price, but simulates an unreliable external
    dependency (e.g. a flaky network call) that fails on its first attempt.
    Use this exactly like lookup_product_price."""
    if _flaky_state["should_fail"]:
        _flaky_state["should_fail"] = False  # fail once, then succeed on retry
        raise ToolException(f"Simulated transient failure looking up '{product_name}'. Please try again.")
    return lookup_product_price.func(product_name)


# Without this line, the ToolException above would propagate and crash
# AgentExecutor.invoke() instead of being handed back to the model as an
# observation it can react to.
flaky_lookup_product_price.handle_tool_error = True

print("Flaky tool defined. handle_tool_error =", flaky_lookup_product_price.handle_tool_error)


Flaky tool defined. handle_tool_error = True


In [21]:
flaky_tools = [flaky_lookup_product_price]
flaky_agent = create_tool_calling_agent(llm, flaky_tools, prompt)
flaky_executor = AgentExecutor(
    agent=flaky_agent,
    tools=flaky_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=6,
)

_flaky_state["should_fail"] = True  # reset so this cell is repeatable

flaky_result = flaky_executor.invoke({"input": "What is the price of the UltraBook Air 13?"})
print("\nFINAL OUTPUT:", flaky_result["output"])




> Entering new AgentExecutor chain...

Invoking: `flaky_lookup_product_price` with `{'product_name': 'UltraBook Air 13'}`


Simulated transient failure looking up 'UltraBook Air 13'. Please try again.
Invoking: `flaky_lookup_product_price` with `{'product_name': 'UltraBook Air 13'}`


{"success": true, "matches": [{"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0, "ram_gb": 8, "storage_gb": 256, "rating": 4.3}]}[{'type': 'text', 'text': 'The price of the UltraBook Air 13 is $899.', 'index': 0, 'extras': {'signature': 'EjQKMgERTTIPvU2heqjUp+MTJ1vXITUKy8vjod+733WjPISnekn8QMxdRVFa/RtL/TlvrLbo'}}]

> Finished chain.

FINAL OUTPUT: [{'type': 'text', 'text': 'The price of the UltraBook Air 13 is $899.', 'index': 0, 'extras': {'signature': 'EjQKMgERTTIPvU2heqjUp+MTJ1vXITUKy8vjod+733WjPISnekn8QMxdRVFa/RtL/TlvrLbo'}}]


**What happened above:** the first call to `flaky_lookup_product_price` raised a
`ToolException`. Because `handle_tool_error=True`, LangChain caught it and fed the
exception's message back to the model as the tool's observation exactly like a
normal tool result, just with `success` implied false by the wording. The model then
retried the same tool, which succeeded on the second attempt, and produced a normal
final answer. No custom retry logic was written `handle_tool_error=True` was the
entire fix.

Compare this to Day 1: there, *every* tool executor already returned a `{"success":
False, "error": ...}` dict instead of raising, so this failure mode never had a chance
to crash the loop. `handle_tool_error` is LangChain's version of that same discipline,
just opt in per tool instead of mandatory by convention.


### What LangChain made easier, and where the magic leaks

LangChain removed real boilerplate: tool schemas are inferred from type hints and
docstrings instead of hand-written twice, the ReAct loop and its `max_iterations`
guardrail are built into `AgentExecutor`, and memory is a drop in wrapper instead of a
list we thread through every call ourselves. But the abstractions do leak in specific
spots. We needed the separate `langchain-classic` package because the exact
constructors this task asks for (`create_tool_calling_agent`, `AgentExecutor`,
`ConversationBufferMemory`) were moved out of core `langchain` in favor of a
LangGraph-based `create_agent`, so "the standard way to build an agent" is already a
moving target across versions. `RunnableWithMessageHistory` prints a deprecation
warning pointing at LangGraph persistence even while still working correctly. And
`handle_tool_error` only catches LangChain's own `ToolException` a bare
`Exception` still crashes the executor unless the tool catches it first, which is the
exact same discipline Day 1 required, just easier to forget because the framework
*looks* like it's handling errors for you everywhere else.


In [22]:
print("\n--- Start chatting with the agent (type 'exit' to quit) ---\n")

chat_session_id = "client-42" # Using the same session ID as before

while True:
    user_input = input("You: ")
    if user_input.lower() == 'exit':
        print("Exiting chat.")
        break

    try:
        agent_response = run_with_memory(chat_session_id, user_input)
        print(f"Agent: {agent_response}")
    except Exception as e:
        print(f"An error occurred: {e}")
        print("Please try again or type 'exit' to quit.")


--- Start chatting with the agent (type 'exit' to quit) ---

You: what is price of ultrabook 12


> Entering new AgentExecutor chain...

Invoking: `lookup_product_price` with `{'product_name': 'ultrabook 12'}`


{"success": false, "error": "No product matching 'ultrabook 12' found in catalog."}[{'type': 'text', 'text': 'I couldn\'t find a product matching "UltraBook 12" in the catalog. The available models are the UltraBook Air 13 and UltraBook Pro 14.', 'index': 0, 'extras': {'signature': 'EjQKMgERTTIPHhTPIxNLHbuAAX6fHr5fiRcMT90feeC6AMIow18oTQW3mm1g56nDTx7yKoO8'}}]

> Finished chain.
Agent: I couldn't find a product matching "UltraBook 12" in the catalog. The available models are the UltraBook Air 13 and UltraBook Pro 14.
You: what is price of ultrabook 13 air


> Entering new AgentExecutor chain...

Invoking: `lookup_product_price` with `{'product_name': 'ultrabook 13 air'}`


{"success": false, "error": "No product matching 'ultrabook 13 air' found in catalog."}
Invoking: `lookup_p